# AutoML с PyCaret
PyCaret автоматически перебирает ~15 классификаторов и ~15 регрессоров, подбирает гиперпараметры победителя и строит
ансамбли (blending, stacking). Это **независимая проверка** ручного выбора модели в `nrg/train.py`:
те же признаки и тот же тест по времени (обучение до 2012, тест 2013–2016).

PyCaret 3.3 работает только на Python ≤ 3.11, поэтому ноутбук выполняется в отдельном окружении `.venv-automl`.

In [1]:
import sys, pandas as pd
from sklearn.metrics import average_precision_score
print(sys.version.split()[0])
train = pd.read_csv('../data/processed/train.csv'); test = pd.read_csv('../data/processed/test.csv')
CAT = ['console', 'rating', 'publisher_main']
train.shape, test.shape, round(train.negative.mean(), 3), round(test.negative.mean(), 3)

3.11.9


((7149, 34), (991, 34), 0.246, 0.142)

## 1. Классификация: провал (оценка < 60) или нет

In [2]:
from pycaret import classification as C
C.setup(data=train.drop(columns=['score']), test_data=test.drop(columns=['score']), target='negative',
        categorical_features=CAT, max_encoding_ohe=40, normalize=True, index=False, fold_strategy='stratifiedkfold',
        fold=5, fold_shuffle=True, session_id=42, verbose=False, html=False, n_jobs=-1)
C.add_metric('pr_auc', 'PR-AUC', average_precision_score, target='pred_proba')
best3 = C.compare_models(sort='PR-AUC', n_select=3, exclude=['lda', 'qda', 'dummy', 'svm', 'ridge'], verbose=False)
C.pull()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,PR-AUC,TT (Sec)
rf,Random Forest Classifier,0.7944,0.7997,0.3639,0.6488,0.4653,0.3503,0.3733,0.5725,0.624
et,Extra Trees Classifier,0.7867,0.7801,0.3553,0.6203,0.4502,0.3298,0.3505,0.5273,0.100
gbc,Gradient Boosting Classifier,0.7686,0.7600,0.2416,0.5748,0.3394,0.2263,0.2581,0.4982,0.196
ada,Ada Boost Classifier,0.7626,0.7445,0.2490,0.5393,0.3404,0.2189,0.2434,0.4686,0.092
lr,Logistic Regression,0.7643,0.7439,0.2137,0.5543,0.3082,0.1987,0.2317,0.4681,1.014
nb,Naive Bayes,0.3602,0.6847,0.9397,0.2701,0.4196,0.0605,0.1362,0.3929,0.794
dt,Decision Tree Classifier,0.7335,0.6488,0.4770,0.4609,0.4685,0.2909,0.2911,0.3503,0.742
knn,K Neighbors Classifier,0.7362,0.6470,0.2678,0.4405,0.3330,0.1807,0.1894,0.3469,0.812


**compare_models** обучил каждую модель на 5 фолдах и отсортировал по PR-AUC (метрика для редкого класса).

In [3]:
tuned = C.tune_model(best3[0], optimize='PR-AUC', n_iter=25, choose_better=True, verbose=False)
blended = C.blend_models(best3, method='soft', choose_better=False, verbose=False)
stacked = C.stack_models(best3, choose_better=False, verbose=False)
te = test.drop(columns=['score'])
rows = []
for name, m in [('лучшая модель', best3[0]), ('после tune_model', tuned), ('blend top-3', blended), ('stack top-3', stacked)]:
    p = C.predict_model(m, data=te.drop(columns=['negative']), raw_score=True, verbose=False)['prediction_score_1']
    rows.append({'вариант': name, 'test PR-AUC': round(average_precision_score(te.negative, p), 3)})
pd.DataFrame(rows)

,вариант,test PR-AUC
0,лучшая модель,0.297
1,после tune_model,0.297
2,blend top-3,0.292
3,stack top-3,0.312


## 2. Регрессия: прогноз самой оценки 0–100

In [4]:
from pycaret import regression as R
R.setup(data=train.drop(columns=['negative']), test_data=test.drop(columns=['negative']), target='score',
        categorical_features=CAT, max_encoding_ohe=40, normalize=True, index=False, fold=5, fold_shuffle=True,
        session_id=42, verbose=False, html=False, n_jobs=-1)
best_r = R.compare_models(sort='MAE', n_select=3, exclude=['dummy'], verbose=False)
R.pull()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,7.782600e+00,1.091715e+02,1.044240e+01,4.419000e-01,0.1764,1.326000e-01,0.266
rf,Random Forest Regressor,8.055200e+00,1.132398e+02,1.063500e+01,4.213000e-01,0.1791,1.370000e-01,0.278
gbr,Gradient Boosting Regressor,9.049300e+00,1.320488e+02,1.148480e+01,3.250000e-01,0.1924,1.542000e-01,0.190
huber,Huber Regressor,9.453300e+00,1.459639e+02,1.207760e+01,2.533000e-01,0.2020,1.634000e-01,0.044
br,Bayesian Ridge,9.518000e+00,1.445075e+02,1.201760e+01,2.607000e-01,0.1998,1.618000e-01,0.030
ridge,Ridge Regression,9.520900e+00,1.447349e+02,1.202710e+01,2.595000e-01,0.1998,1.618000e-01,0.030
lar,Least Angle Regression,9.520900e+00,1.447374e+02,1.202720e+01,2.595000e-01,0.1998,1.618000e-01,0.030
omp,Orthogonal Matching Pursuit,9.640700e+00,1.485021e+02,1.218060e+01,2.407000e-01,0.2023,1.641000e-01,0.032
lasso,Lasso Regression,9.779800e+00,1.513881e+02,1.229810e+01,2.261000e-01,0.2047,1.671000e-01,0.034
llar,Lasso Least Angle Regression,9.779800e+00,1.513881e+02,1.229810e+01,2.261000e-01,0.2047,1.671000e-01,0.028


**Вывод.** PyCaret подтверждает ручной выбор: лидируют ансамбли деревьев (Random Forest / Extra Trees / бустинг);
подбор гиперпараметров и комбинации дают небольшой прирост. Регрессия ошибается в среднем на ~8 баллов из 100.